# Lab | Summarization evaluation using LangSmith
Let's revisit your capstone project 2? Well, sort of. Pick diffierent sets of data and re-run this notebook. Maybe parts of the dataset you used in your last project week. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications using LangSmith.

What did you learn? - Let's discuss that in class

## LangSmith - LangChain evaluation

In [ ]:
from dotenv import load_dotenv, find_dotenv
import os

DOTENV_PATH = r"E:\AI\Ironhack\Leasons\.env"

if os.path.exists(DOTENV_PATH):
    load_dotenv(DOTENV_PATH, override=True)
else:
    load_dotenv(find_dotenv(), override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")
LANGCHAIN_API_KEY = LANGSMITH_API_KEY
HUGGINGFACEHUB_API_TOKEN = (
    os.getenv("HUGGINGFACEHUB_API_TOKEN")
    or os.getenv("HUGGING_FACE_HUB_TOKEN")
    or os.getenv("HF_TOKEN")
)

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if LANGSMITH_API_KEY:
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_API_KEY
if HUGGINGFACEHUB_API_TOKEN:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACEHUB_API_TOKEN

In [ ]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "langsmith-summarization-lab"
os.environ["LANGCHAIN_PROJECT"] = "langsmith-summarization-lab"

In [ ]:
#Importing Client from Langsmith
from langsmith import Client
client = Client(api_key=LANGSMITH_API_KEY)

### Create Dataset


In [ ]:
from datasets import load_dataset

cnn_dataset = load_dataset(
    "ccdv/cnn_dailymail",
    "3.0.0",
    trust_remote_code=True
)

In [ ]:
def add_prefix(example):
    return {
        **example,
        "article": f"Summarize this news:\n{example['article']}"
    }

#cnn_dataset = cnn_dataset.map(add_prefix)

In [ ]:
cnn_dataset

In [ ]:
cnn_dataset['train'][0]

In [ ]:
#Get just a few news to test
MAX_NEWS=10
sample_cnn = cnn_dataset["test"].select(range(MAX_NEWS)).map(add_prefix)

sample_cnn

The dataset contains three columns: article, highlights, and id. To use LangSmith, we need to create a dataset in LangSmith format.

LangSmith expects a prompt and a result. To achieve this, we will transform the article into a prompt by adding the prefix: "Summarize this news." As a result, we will use the content of highlights, which represents the summaries created by humans.

In [ ]:
print(sample_cnn[0])

Now We have the Dataset with the prompt and the Reference Summary, it is time to create a Dataset in LangSmith with this information.
### Create the Dataset in Langsmith

The dataset in LangSmith is composed of an input, which is the prompt passed to the model for evaluation, and an output, which should contain what we expect the model to return.

In [ ]:
import datetime

In [ ]:
import uuid
input_key = ['article']
output_key = ['highlights']

NAME_DATASET = f"Summarize_dataset_{datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"

In [ ]:
#This creates the dataset in LangSmith with the content in sample_cnn - If you run this more than once you will get POST errors
sample_cnn_df = sample_cnn.to_pandas()

dataset = client.upload_dataframe(
    df=sample_cnn_df,
    input_keys=input_key,
    output_keys=output_key,
    name=NAME_DATASET,
    description="Test Embedding distance between model summarizations",
    data_type="kv"
)

In this image, we can see an example from the dataset once it's been registered in LangSmith.

In the Input column, there is the prompt to be sent, while in the Output column, the expected output is stored.

When performing the comparison, the model will be given the prompt, and the Cosine distance between its response and the one stored in the sample dataset will be calculated.
<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Dataset.jpg?raw=true">

### Recovering Models From Hugging Face
Let's retrieve both models from HuggingFace. A base T5 model and a model that has been fine-tuned using the training portion of this same dataset to generate summaries.

In [ ]:
try:
    from langchain_huggingface import HuggingFaceEndpoint
except ImportError:
    HuggingFaceEndpoint = None
    try:
        from langchain_community.llms import HuggingFaceHub
    except ImportError:
        from langchain import HuggingFaceHub

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

summary_prompt = PromptTemplate.from_template("{article}")

In [ ]:
if HuggingFaceEndpoint is not None:
    summarizer_base = HuggingFaceEndpoint(
        repo_id="t5-base",
        task="text2text-generation",
        max_new_tokens=180,
        do_sample=False,
        huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN,
    )
else:
    summarizer_base = HuggingFaceHub(
        repo_id="t5-base",
        model_kwargs={"temperature": 0, "max_length": 180},
        huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN
    )

summarizer_base_chain = summary_prompt | summarizer_base | StrOutputParser()

In [ ]:
if HuggingFaceEndpoint is not None:
    summarizer_finetuned = HuggingFaceEndpoint(
        repo_id="flax-community/t5-base-cnn-dm",
        task="text2text-generation",
        max_new_tokens=180,
        do_sample=False,
        huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN,
    )
else:
    summarizer_finetuned = HuggingFaceHub(
        repo_id="flax-community/t5-base-cnn-dm",
        model_kwargs={"temperature": 0, "max_length": 180},
        huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN
    )

summarizer_finetuned_chain = summary_prompt | summarizer_finetuned | StrOutputParser()

## Defining Evaluator
The first step is to define an evaluator, where we specify the variables we want to evaluate. In our case, I have chosen to measure only the "embedding_distance."

I've left the "string_distance" as a comment in case you want to conduct a test with two evaluations instead of one.


In [ ]:
try:
    from langchain_classic.smith.evaluation import run_on_dataset, RunEvalConfig
except ImportError:
    from langchain.smith import run_on_dataset, RunEvalConfig

# !pip install -q rapidfuzz==3.6.1

In [ ]:
#We are using just one of the multiple evaluator avaiable on LangSmith.

evaluation_config = RunEvalConfig(
    evaluators=[
        "embedding_distance",
        #"string_distance"
    ],
    input_key="article",
    reference_key="highlights",
    prediction_key="output",
)

### Running Evaluator
With the same configuration, we can launch two evaluations on the same dataset. One for each of the chosen models.

In [ ]:
project_name = f"T5-BASE {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

base_t5_results = run_on_dataset(
    client=client,
    project_name=project_name,
    dataset_name=NAME_DATASET,
    llm_or_chain_factory=lambda: summarizer_base_chain,
    evaluation=evaluation_config,
)

In [ ]:
#Ignore the error shown below
project_name = f"T5-FineTuned {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

finetuned_t5_results = run_on_dataset(
    client=client,
    project_name=project_name,
    dataset_name=NAME_DATASET,
    llm_or_chain_factory=lambda: summarizer_finetuned_chain,
    evaluation=evaluation_config,
)

<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Tests.jpg?raw=true">

In the image below you can see the comparision between two tests.
<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_CompareTestst.jpg?raw=true">

Well, since it has been so straightforward, why don't we try to make the comparison with an OpenAI model?

In [ ]:
from langchain_openai import ChatOpenAI

open_aillm = ChatOpenAI(model="gpt-5.4-mini", temperature=0.0)
open_ai_chain = summary_prompt | open_aillm | StrOutputParser()

In [ ]:
project_name = f"OpenAI {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

openai_results = run_on_dataset(
    client=client,
    project_name=project_name,
    dataset_name=NAME_DATASET,
    llm_or_chain_factory=lambda: open_ai_chain,
    evaluation=evaluation_config,
)

<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_CompareOpenAI_HF.jpg?raw=true">

The experiment with the OpenAI model has yielded the best results. But, be aware! As we can see, there is a cost involved since we are using an API, and it needs to be paid for.

Another crucial piece of information is that we can view performance data for the models. This data could also be useful for minimally evaluating our inference server.

### What I learned
In this lab I learned how to create a small evaluation dataset in LangSmith, run the same summarization task with different models, and compare their answers with an evaluator. The main idea is that we should not only check if a model gives an answer, but also measure how close the answer is to the reference summary. I also saw that OpenAI can give strong results, but it uses a paid API, so model quality and cost both matter.